# Verified reference run — published BNN experiment

This executed notebook documents a verified reproduction of the numerical experiment reported in **Predicting Arrival Delays and Flight Time Deviations Under Uncertainty Using Bayesian Neural Networks** (IEEE Access, 2026; DOI: 10.1109/ACCESS.2026.3709713).

It preserves the paper-faithful preprocessing, split, training schedule, optimizer behavior, RNG sequence, model-construction order, and Monte Carlo evaluation used by the archived executable implementation.

The public route-specific dataset contains exactly 14,956 SBRF-SBGR records. After the source-code drops, the executable experiment uses 23 inputs, including wind direction and gust speed but not sustained wind speed.


In [1]:

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
# Preserve the single-thread CPU numerics used in the verified reproduction.
torch.set_num_threads(1)
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.preprocessing import MinMaxScaler

SEED = 42
candidates = [
    Path("../data/final_data_sbrf_sbgr.csv.gz"),
    Path("data/final_data_sbrf_sbgr.csv.gz"),
    Path("../data/final_data_sbrf_sbgr.csv"),
    Path("data/final_data_sbrf_sbgr.csv"),
]
DATASET = next((p for p in candidates if p.exists()), candidates[0])

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reset_seed()
print("Dataset:", DATASET)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Dataset: final_data_sbgr.csv
PyTorch: 2.10.0+cpu
CUDA available: False


In [2]:

raw = pd.read_csv(DATASET)
print("Raw shape:", raw.shape)
df = raw[raw["sg_icao_origem"] == 2].copy().reset_index(drop=True)
print("SBRF-SBGR shape:", df.shape)


Raw shape: (254183, 28)
SBRF-SBGR shape: (14956, 28)


In [3]:

OUTPUT_COLUMN = "arrival_delay"
df["flight_time_deviation_clipped"] = np.maximum(df["flight_time_deviation"], 0)
df["arrival_delay_clipped"] = np.maximum(df["arrival_delay"], 0)
alternate_outputs = [
    "flight_time_deviation", "flight_time_deviation_clipped",
    "arrival_delay", "arrival_delay_clipped",
]
alternate_outputs.remove(OUTPUT_COLUMN)
for col in alternate_outputs:
    if col in df.columns:
        df = df.drop(columns=col)
df = df.drop(columns=["sg_icao_origem", "tmi_duration", "rate_nm"])
for col in list(df.columns):
    if col != OUTPUT_COLUMN and (df[col] == 0).all():
        df = df.drop(columns=col)
cols = [c for c in df.columns if c != OUTPUT_COLUMN] + [OUTPUT_COLUMN]
df = df[cols]
print("Modeling shape:", df.shape)
print("Input features:", df.shape[1] - 1)
print(list(df.columns[:-1]))


Modeling shape: (14956, 24)
Input features: 23
['drct', 'vsby', 'ceiling', 'tmpf', 'dwpf', 'relh', 'alti', 'gust_speed', 'ifr', 'lifr', 'ts', 'feel', 'arrival_demand', 'sg_icao_empresa', 'model_equip', 'scheduled_flight_time', 'adjusted_arrival_time_sin', 'adjusted_arrival_time_cos', 'adjusted_arrival_day_sin', 'adjusted_arrival_day_cos', 'adjusted_arrival_month_sin', 'adjusted_arrival_month_cos', 'departure_delay']


In [4]:

x_full = df.iloc[:, :-1]
y_full = df.iloc[:, -1]
x_full, y_full = shuffle(x_full, y_full, random_state=SEED)
x_train_val, x_test, y_train_val, y_test = train_test_split(
    x_full, y_full, test_size=1000, random_state=SEED
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_val, y_train_val, train_size=10000, random_state=SEED
)
print("Train/validation/test:", len(x_train), len(x_val), len(x_test))


Train/validation/test: 10000 3956 1000


In [5]:

scaler = MinMaxScaler()
train_array = np.column_stack([x_train.values, y_train.values])
val_array = np.column_stack([x_val.values, y_val.values])
test_array = np.column_stack([x_test.values, y_test.values])
train_scaled = scaler.fit_transform(train_array)
val_scaled = scaler.transform(val_array)
test_scaled = scaler.transform(test_array)
x_train_t = torch.tensor(train_scaled[:, :-1], dtype=torch.float32)
y_train_t = torch.tensor(train_scaled[:, -1:], dtype=torch.float32)
x_val_t = torch.tensor(val_scaled[:, :-1], dtype=torch.float32)
y_val_t = torch.tensor(val_scaled[:, -1:], dtype=torch.float32)
x_test_t = torch.tensor(test_scaled[:, :-1], dtype=torch.float32)
y_test_t = torch.tensor(test_scaled[:, -1:], dtype=torch.float32)


In [6]:

class BNN(nn.Module):
    def __init__(self, n_inputs, hidden=128, prior_sigma=1e-2, output_prior_sigma=1e-1):
        super().__init__()
        self.net = nn.Sequential(
            bnn.BayesLinear(prior_mu=0, prior_sigma=prior_sigma, in_features=n_inputs, out_features=hidden),
            nn.ReLU(),
            bnn.BayesLinear(prior_mu=0, prior_sigma=prior_sigma, in_features=hidden, out_features=hidden),
            nn.ReLU(),
            bnn.BayesLinear(prior_mu=0, prior_sigma=output_prior_sigma, in_features=hidden, out_features=1),
        )
    def forward(self, x):
        return self.net(x)

class ANN(nn.Module):
    def __init__(self, n_inputs, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_inputs, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x)


In [7]:

reset_seed()
model = BNN(x_train_t.shape[1], hidden=128, prior_sigma=1e-2, output_prior_sigma=1e-1)
_ann_model = ANN(x_train_t.shape[1], hidden=128)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=200)
mse_loss_fn = nn.MSELoss()
kl_loss_fn = bnn.BKLLoss(reduction="mean", last_layer_only=False)
final_kl_weight = 10 / len(x_train_t)


In [8]:

best_val_loss = float("inf")
patience_counter = 0
patience = 1000
best_model_state = model.state_dict().copy()
final_epoch = 0
for epoch in range(5000):
    model.train()
    pred = model(x_train_t)
    mse = mse_loss_fn(pred, y_train_t)
    optimizer.zero_grad(); mse.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = mse_loss_fn(model(x_val_t), y_val_t).item()
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss; patience_counter = 0; best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    final_epoch = epoch
    if patience_counter >= patience:
        break
model.load_state_dict(best_model_state)
print("Warmup final epoch:", final_epoch)


Warmup final epoch: 1322


In [9]:

for epoch in range(final_epoch, 10000):
    model.train()
    pred = model(x_train_t)
    mse = mse_loss_fn(pred, y_train_t)
    kl = kl_loss_fn(model)
    loss = mse + final_kl_weight * kl
    optimizer.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()


In [10]:

model.eval()
with torch.no_grad():
    _train_samples = np.array([model(x_train_t).detach().numpy() for _ in range(1000)])
mse_mean = 0.0
with torch.no_grad():
    for _ in range(1000):
        mse_mean += mse_loss_fn(model(x_test_t), y_test_t).item()
mse_mean /= 1000
with torch.no_grad():
    test_samples = np.array([model(x_test_t).detach().numpy() for _ in range(1000)])
test_samples = test_samples[:, :, 0].T
y_min = scaler.data_min_[-1]; y_max = scaler.data_max_[-1]
samples_unscaled = y_min + test_samples * (y_max - y_min)
mean_values = samples_unscaled.mean(axis=1); std_values = samples_unscaled.std(axis=1)
y_test_original = y_test.values
mae = np.mean(np.abs(mean_values - y_test_original))
coverage_95 = np.mean((y_test_original >= mean_values - 2*std_values) & (y_test_original <= mean_values + 2*std_values))
print("Scaled test MSE:", mse_mean)
print("Test MAE:", mae, "min")
print("95% coverage:", 100*coverage_95, "%")
print("Mean sigma:", std_values.mean(), "min")


Scaled test MSE: 0.007269820394910872
Test MAE: 6.676429349303692 min
95% coverage: 97.3 %
Mean sigma: 9.972439190254652 min
